# Phase 1 — InfoPrune: entropy-guided adaptive-layer pruning

**Builds on your working FastV Phase-0 notebook** — same model (`llava-hf/llava-1.5-7b-hf`), same
`fastv_config` hook, same POPE subset. This notebook assumes you're continuing in that Kaggle session
(so `MODEL_ID`, `processor`, `bnb_config`, `img_dir`, `subset`/`all_questions` from Phase 0 are still
in memory). If you're starting fresh, re-run Phase 0's Sections 1-4 first (env check, install, smoke
test, POPE data prep) before this notebook's cells.

**What this does, in one paragraph:** instead of FastV's fixed pruning layer `K` (same for every
image), we track a cheap entropy-based "reliability" score at each layer as the forward pass runs,
and prune at the *first* layer whose score crosses a threshold — calibrated once, offline, on a small
image set. This is the InfoScore idea from the Segmentation-From-Attention paper, adapted from
"which layer segments best" to "which layer's attention is trustworthy enough to prune on."

**What I need from you:** run Section 2 (calibration) and Section 3 (smoke test) first — those are
cheap and will immediately tell us if the entropy tracking behaves sensibly before we spend GPU time
on the full POPE comparison in Section 4. Paste back the printed calibration stats and smoke-test
trace even if nothing errors, since I want to sanity-check the actual numbers, not just whether it
runs.


## Environment Check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory/1e9:.1f} GB, compute capability {props.major}.{props.minor}")


## FastV Install

In [ ]:
%%bash
cd /kaggle/working
if [ ! -d "FastV" ]; then
  git clone --depth 1 https://github.com/pkunlp-icler/FastV.git
fi
cd FastV/src/FastV/llava-hf/transformers
pip install -e . --no-deps -q
pip install -q "tokenizers<0.19,>=0.14" pillow accelerate bitsandbytes sentencepiece
python -c "import transformers; print('transformers version:', transformers.__version__)"

## Smoke Test

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [1]:
import torch
# from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from transformers import (
    AutoTokenizer,             # <-- Added
    CLIPImageProcessor,        # <-- Added
    LlavaProcessor,            # <-- Added
    LlavaForConditionalGeneration,
    BitsAndBytesConfig
)
from PIL import Image
import requests

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
# processor = AutoProcessor.from_pretrained(MODEL_ID)

# --- THE FIX ---
# Load the components individually to bypass the updated config file
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
image_processor = CLIPImageProcessor.from_pretrained(MODEL_ID)
processor = LlavaProcessor(image_processor=image_processor, tokenizer=tokenizer)
# ----

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

In [2]:
prompt = "USER: <image>\nWhat is shown in this image? Answer in one sentence.\nASSISTANT:"
image = Image.open(requests.get("http://images.cocodataset.org/val2017/000000039769.jpg", stream=True).raw)

def load_model(fastv_config=None):
    kwargs = dict(quantization_config=bnb_config, attn_implementation="eager", device_map="auto")
    if fastv_config is not None:
        kwargs["fastv_config"] = fastv_config
    return LlavaForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs)

print("Loading baseline (no FastV)...")
model = load_model(fastv_config=None)
inputs = processor(prompt, image, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=64, do_sample=False, use_cache=False)
print("BASELINE:", processor.decode(out[0], skip_special_tokens=True))
del model
torch.cuda.empty_cache()

fastv_config = {
    "use_fastv": True,
    "fastv_k": 2,
    "fastv_r": 0.5,
    "image_token_start_index": 5,
    "image_token_length": 576,
}
print("\nLoading FastV (K=2, R=50%)...")
model = load_model(fastv_config=fastv_config)
inputs = processor(prompt, image, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=64, do_sample=False, use_cache=False)
print("FASTV:   ", processor.decode(out[0], skip_special_tokens=True))
del model
torch.cuda.empty_cache()

Loading baseline (no FastV)...


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

BASELINE: USER:  <image> 
What is shown in this image? Answer in one sentence.
ASSISTANT: Two cats sleeping on a couch.

Loading FastV (K=2, R=50%)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

FASTV:    USER:  <image> 
What is shown in this image? Answer in one sentence.
ASSISTANT: Two cats sleeping on a couch.


## POPE -- Data Prep

In [3]:
%%bash
mkdir -p /kaggle/working/data
cd /kaggle/working/data
if [ ! -d "POPE" ]; then
  git clone --depth 1 https://github.com/AoiDragon/POPE.git
fi
ls POPE/output/coco

coco_pope_adversarial.json
coco_pope_popular.json
coco_pope_random.json


Cloning into 'POPE'...


In [4]:
import json, random, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import urllib.request

random.seed(0)
N_PER_SPLIT = 150  # ~150 questions per POPE split (random/popular/adversarial) = 450 total. Raise once the pipeline is confirmed working.

pope_dir = Path("/kaggle/working/data/POPE/output/coco")
splits = ["coco_pope_random.json", "coco_pope_popular.json", "coco_pope_adversarial.json"]

subset = {}
for split in splits:
    lines = [json.loads(l) for l in open(pope_dir / split) if l.strip()]
    # sample by unique image to keep the image-download set small, then take all questions for those images
    images = sorted(set(l["image"] for l in lines))
    random.shuffle(images)
    keep_images = set(images[: max(1, N_PER_SPLIT // 6)])  # ~6 questions/image on average in POPE
    sampled = [l for l in lines if l["image"] in keep_images][:N_PER_SPLIT]
    subset[split] = sampled
    print(split, "->", len(sampled), "questions over", len(keep_images), "images")

img_dir = Path("/kaggle/working/data/coco_val2014_subset")
img_dir.mkdir(exist_ok=True)
all_images = sorted(set(l["image"] for s in subset.values() for l in s))
print(f"\nDownloading {len(all_images)} unique COCO val2014 images...")

def fetch(fname):
    dest = img_dir / fname
    if dest.exists():
        return
    url = f"http://images.cocodataset.org/val2014/{fname}"
    urllib.request.urlretrieve(url, dest)

with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(fetch, all_images))

print("Done. Downloaded:", len(list(img_dir.glob('*.jpg'))))


coco_pope_random.json -> 150 questions over 25 images
coco_pope_popular.json -> 150 questions over 25 images
coco_pope_adversarial.json -> 150 questions over 25 images

Done. Downloaded: 73


## 1. The InfoPrune module

This is a from-scratch adaptation of FastV's `fastv_forward` (same file you already have installed:
`FastV/src/FastV/llava-hf/transformers/.../modeling_llama.py`) — I kept the pruning mechanics
(top-k attention, index-based token removal) identical to FastV, and replaced the fixed-`K` decision
with the entropy-based stopping rule. It's a **monkeypatch**, not an edit to the installed package
file — it overwrites `LlamaModel.fastv_forward` at runtime, so you can flip between plain FastV and
InfoPrune within the same notebook just by calling `enable_infoprune()` / `disable_infoprune()`.

**Known limitation (matches a bug already present in FastV's own code):** their KV-cache pruning path
has a comment in the source — `# lead the cuda error in the second iteration of decoding layeridx 3`
— i.e. it's not fully solid. Since your Phase-0 notebook already ran everything with
`use_cache=False`, this module only supports that mode for now. Full KV-cache support is a Phase-3/4
engineering item if it turns out to matter for the paper's efficiency numbers, not something to chase
right now.

In [10]:
import bisect

def compute_relevant_text_tokens(embed_hidden_states, v_token_start, v_token_len, text_token_start):
    """SparseVLM-style text-token relevance ('Select Text Raters'): dot-product similarity between
    raw (pre-layer-0) image-patch and text-token embeddings, softmax over the text dimension, averaged
    over image patches, thresholded at the mean. Returns absolute sequence indices of 'relevant' text
    tokens. embed_hidden_states: (1, seq_len, hidden_dim) -- the embedding layer's output, i.e.
    hidden_states before layer 0, or equivalently out.hidden_states[0] with output_hidden_states=True."""
    v_t = embed_hidden_states[:, v_token_start: v_token_start + v_token_len, :]
    t_t = embed_hidden_states[:, text_token_start:, :]
    if t_t.shape[1] == 0:
        return torch.tensor([], dtype=torch.long, device=embed_hidden_states.device)
    m_v_t = v_t @ t_t.transpose(1, 2)            # (1, V, T)
    m_v_t = m_v_t.softmax(dim=2).mean(dim=1)[0]  # (T,)
    idx = torch.where(m_v_t > m_v_t.mean())[0]
    return idx + text_token_start

In [31]:
import torch
import transformers.models.llama.modeling_llama as modeling_llama
from transformers.modeling_outputs import BaseModelOutputWithPast

# Keep a handle to the original fixed-K FastV forward so we can toggle back to it for comparisons.
_ORIGINAL_FASTV_FORWARD = modeling_llama.LlamaModel.fastv_forward

# Side channel for chosen_layer/trace: model.generate() wraps this forward's output in a new
# LlavaCausalLMOutputWithPast, dropping any custom attributes we attach to our own output object.
# Simplest robust fix: write results here as a side effect instead of relying on attribute pass-through.
# Call `_LAST_TRACE["chosen_layer"] = None; _LAST_TRACE["trace"] = None` before each generate() call,
# then read it back after -- see Sections 3 and 4 below.
_LAST_TRACE = {"chosen_layer": None, "trace": None}


import bisect

def compute_relevant_text_tokens(embed_hidden_states, v_token_start, v_token_len, text_token_start):
    """SparseVLM-style text-token relevance ('Select Text Raters'): dot-product similarity between
    raw (pre-layer-0) image-patch and text-token embeddings, softmax over the text dimension, averaged
    over image patches, thresholded at the mean. Returns absolute sequence indices of 'relevant' text
    tokens. embed_hidden_states: (1, seq_len, hidden_dim) -- the embedding layer's output."""
    v_t = embed_hidden_states[:, v_token_start: v_token_start + v_token_len, :]
    t_t = embed_hidden_states[:, text_token_start:, :]
    if t_t.shape[1] == 0:
        return torch.tensor([], dtype=torch.long, device=embed_hidden_states.device)
    m_v_t = v_t @ t_t.transpose(1, 2)            # (1, V, T)
    m_v_t = m_v_t.softmax(dim=2).mean(dim=1)[0]  # (T,)
    idx = torch.where(m_v_t > m_v_t.mean())[0]
    return idx + text_token_start


def _entropy_over_image_tokens(attn_weights, image_start, image_len, text_token_idx, eps=1e-8):
    """attn_weights: (batch, heads, seq_q, seq_k) from one decoder layer, output_attentions=True.
    Query row = average attention over the SparseVLM-selected 'relevant' text-token positions
    (falls back to the last token only if none were marked relevant -- e.g. a degenerate/very
    short prompt). Clamped to non-negative before the entropy calc (see earlier NaN fix note)."""
    text_token_idx = text_token_idx.to(attn_weights.device)
    avg_heads = attn_weights.mean(dim=1)[0]                 # (seq_q, seq_k)
    if text_token_idx.numel() == 0:
        query_row = avg_heads[-1]
    else:
        query_row = avg_heads[text_token_idx].mean(dim=0)
    image_attn = query_row[image_start: image_start + image_len].clamp(min=0)
    image_attn = image_attn / (image_attn.sum() + eps)
    entropy = -(image_attn * (image_attn + eps).log()).sum()
    return entropy.item(), image_attn


def infoprune_forward(
    self,
    input_ids=None,
    attention_mask=None,
    position_ids=None,
    past_key_values=None,
    inputs_embeds=None,
    use_cache=None,
    output_attentions=None,
    output_hidden_states=None,
    return_dict=None,
    cache_position=None,
    fastv_config=None,
):
    cfg = fastv_config
    image_start = cfg['image_token_start_index']
    image_len = cfg['image_token_length']
    r = cfg['fastv_r']
    layer_floor = cfg.get('layer_floor', 2)
    layer_ceiling = cfg.get('layer_ceiling', 16)
    dataset_entropy = cfg['dataset_entropy']
    layer_calib_scores = cfg['layer_calib_scores']

    use_cache = use_cache if use_cache is not None else self.config.use_cache
    assert not use_cache, "InfoPrune prototype currently supports use_cache=False only."
    output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
    return_dict = return_dict if return_dict is not None else self.config.use_return_dict

    if inputs_embeds is None:
        inputs_embeds = self.embed_tokens(input_ids)

    past_seen_tokens = 0
    if cache_position is None:
        cache_position = torch.arange(past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device)
    if position_ids is None:
        position_ids = cache_position.unsqueeze(0)

    causal_mask = self._update_causal_mask(attention_mask, inputs_embeds, past_seen_tokens)
    hidden_states = inputs_embeds
    all_hidden_states = () if output_hidden_states else None
    seq_length_with_past = past_seen_tokens + inputs_embeds.shape[1]

    # SparseVLM-style relevant-text-token selection, computed once from raw embeddings (matches
    # what the calibration cell must also do -- see the updated calibration cell below).
    text_token_start = image_start + image_len
    if hidden_states.shape[1] != 1:
        text_token_idx = compute_relevant_text_tokens(inputs_embeds, image_start, image_len, text_token_start)
    else:
        text_token_idx = torch.tensor([], dtype=torch.long, device=hidden_states.device)

    pruned = False
    pruned_attention_mask = None
    chosen_layer = None
    candidate_scores = {}
    candidate_image_attn = {}
    is_prefill = hidden_states.shape[1] != 1

    for layer_idx, decoder_layer in enumerate(self.layers):
        if output_hidden_states:
            all_hidden_states += (hidden_states,)

        want_attn = (not pruned) and (layer_floor <= layer_idx <= layer_ceiling) and hidden_states.shape[1] != 1

        layer_outputs = decoder_layer(
            hidden_states,
            attention_mask=causal_mask if not pruned else pruned_attention_mask,
            position_ids=position_ids,
            past_key_value=past_key_values,
            output_attentions=want_attn,
            use_cache=False,
            cache_position=cache_position,
        )
        hidden_states = layer_outputs[0]

        if want_attn and not pruned:
            h_img, image_attn_vec = _entropy_over_image_tokens(layer_outputs[1], image_start, image_len, text_token_idx)
            raw_score = dataset_entropy[layer_idx] / (h_img + 1e-8)
            sorted_calib = layer_calib_scores[layer_idx]
            percentile = bisect.bisect_left(sorted_calib, raw_score) / len(sorted_calib)
            candidate_scores[layer_idx] = percentile
            candidate_image_attn[layer_idx] = image_attn_vec.detach()

            if layer_idx == layer_ceiling:
                best_layer = max(candidate_scores, key=candidate_scores.get)
                chosen_layer = best_layer
                image_scores = candidate_image_attn[best_layer].to(hidden_states.device)
                keep_k = round(image_len * (1 - r))
                top_idx = image_scores.topk(keep_k).indices + image_start
                keep_indexs = torch.cat((
                    torch.arange(image_start, device=top_idx.device),
                    top_idx,
                    torch.arange(image_start + image_len, seq_length_with_past, device=top_idx.device),
                )).sort().values

                hidden_states = hidden_states[:, keep_indexs, :]
                position_ids = keep_indexs.unsqueeze(0)
                pruned_attention_mask = self._update_causal_mask(None, hidden_states, 0)
                cache_position = cache_position[: keep_indexs.shape[0]]
                pruned = True

    hidden_states = self.norm(hidden_states)
    if output_hidden_states:
        all_hidden_states += (hidden_states,)

    if is_prefill and _LAST_TRACE.get("chosen_layer") is None:
        _LAST_TRACE["chosen_layer"] = chosen_layer
        _LAST_TRACE["trace"] = {l: round(s, 4) for l, s in candidate_scores.items()}

    return BaseModelOutputWithPast(
        last_hidden_state=hidden_states,
        past_key_values=None,
        hidden_states=all_hidden_states,
        attentions=None,
    )
def enable_infoprune():
    modeling_llama.LlamaModel.fastv_forward = infoprune_forward

def disable_infoprune():
    modeling_llama.LlamaModel.fastv_forward = _ORIGINAL_FASTV_FORWARD

print("InfoPrune module loaded. Use enable_infoprune() / disable_infoprune() to toggle.")


InfoPrune module loaded. Use enable_infoprune() / disable_infoprune() to toggle.


## 2. Calibration — compute H_data(l) and pick a threshold

Reuses the POPE COCO subset images you already downloaded in Phase 0 (`img_dir`) — no new data
needed. This runs the *unpatched* baseline model (plain `output_attentions=True`, no pruning) once
per calibration image, extracts each candidate layer's attention-over-image-tokens, and:

1. Averages that distribution across all calibration images per layer → `dataset_entropy[l]`
   (high = layer's attention pattern is content-adaptive across images, not a fixed positional bias)
2. Also records the *per-image* InfoScore trace, so we can pick `info_threshold` empirically —
   we choose the threshold value that would make the average calibration image stop at a target
   layer (default: layer 6), rather than guessing a number out of thin air.

This is a one-time cost (~100-150 images, baseline model only, no generation — just a single forward
pass per image) and should run in a few minutes.

In [32]:
import random
from pathlib import Path
from PIL import Image

random.seed(0)
N_CALIB = 120
LAYER_FLOOR, LAYER_CEILING = 2, 16
TARGET_AVG_LAYER = 6  # tune this: lower = more compute saved, higher = closer to baseline accuracy
image_start, image_len = 5, 576   # add this line back
layer_range = list(range(LAYER_FLOOR, LAYER_CEILING + 1))

calib_images = list(img_dir.glob("*.jpg"))
random.shuffle(calib_images)
calib_images = calib_images[:N_CALIB]
print(f"Calibrating on {len(calib_images)} images from {img_dir}")



model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, attn_implementation="eager", device_map = "auto"
)
model.eval()

text_token_start = image_start + image_len

sums = {l: torch.zeros(image_len, device=model.device) for l in layer_range}
counts = {l: 0 for l in layer_range}
per_image_traces = []

for i, path in enumerate(calib_images):
    image = Image.open(path).convert("RGB")
    prompt = "USER: <image>\nDescribe this image.\nASSISTANT:"
    inputs = processor(prompt, image, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**inputs, output_attentions=True, output_hidden_states=True, use_cache=False, return_dict=True)
    embed_output = out.hidden_states[0]   # pre-layer-0 embeddings, needed for relevance scoring
    text_token_idx = compute_relevant_text_tokens(embed_output, image_start, image_len, text_token_start)
    trace = {}
    for l in layer_range:
        h, img_attn = _entropy_over_image_tokens(out.attentions[l], image_start, image_len, text_token_idx)
        sums[l] += img_attn
        counts[l] += 1
        trace[l] = h
    per_image_traces.append(trace)
    if (i + 1) % 30 == 0:
        print(f"  calibrated {i+1}/{len(calib_images)}")

# dataset_entropy and layer_calib_scores computation stays exactly the same as before, just
# reading from this updated per_image_traces / sums.

dataset_entropy = {}
for l in layer_range:
    avg_profile = sums[l] / counts[l]
    avg_profile = avg_profile / avg_profile.sum()
    dataset_entropy[l] = -(avg_profile * (avg_profile + 1e-8).log()).sum().item()

del model
torch.cuda.empty_cache()

print("\nH_data(l) per layer:")
for l in layer_range:
    print(f"  layer {l:2d}: H_data={dataset_entropy[l]:.4f}")

Calibrating on 73 images from /kaggle/working/data/coco_val2014_subset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  calibrated 30/73
  calibrated 60/73

H_data(l) per layer:
  layer  2: H_data=5.1534
  layer  3: H_data=5.4496
  layer  4: H_data=5.5708
  layer  5: H_data=5.5652
  layer  6: H_data=5.3566
  layer  7: H_data=5.4119
  layer  8: H_data=5.3587
  layer  9: H_data=5.4335
  layer 10: H_data=5.3410
  layer 11: H_data=5.6566
  layer 12: H_data=5.6099
  layer 13: H_data=5.7259
  layer 14: H_data=5.5982
  layer 15: H_data=5.6019
  layer 16: H_data=5.5366


In [33]:
import numpy as np
# --- Pick info_threshold so the average calibration image would stop at TARGET_AVG_LAYER ---


# NaN check (fp16 mean-pooling artifact -- see earlier fix)
bad_indices = [i for i, t in enumerate(per_image_traces) if any(np.isnan(v) for v in t.values())]
print(f"{len(bad_indices)} of {len(per_image_traces)} calibration images produced a NaN trace.")
clean_traces = [t for i, t in enumerate(per_image_traces) if i not in bad_indices]
print(f"Proceeding with {len(clean_traces)} clean calibration traces.")

# Percentile-rank calibration stats, per layer
layer_calib_scores = {}
for l in layer_range:
    layer_calib_scores[l] = sorted(dataset_entropy[l] / (t[l] + 1e-8) for t in clean_traces)
    
target_scores = []
for trace in per_image_traces:
    h_img = trace[TARGET_AVG_LAYER]
    target_scores.append(dataset_entropy[TARGET_AVG_LAYER] / (h_img + 1e-8))
info_threshold = float(sorted(target_scores)[len(target_scores) // 2])  # median

print(f"\nChosen info_threshold = {info_threshold:.4f} (targeting avg stop ~ layer {TARGET_AVG_LAYER})")

5 of 73 calibration images produced a NaN trace.
Proceeding with 68 clean calibration traces.

Chosen info_threshold = 1.3993 (targeting avg stop ~ layer 6)


**What to check before continuing:** `H_data(l)` should generally *increase* with layer depth up to
some point (early layers tend to have more positionally-biased, less content-specific attention) —
if it's flat or noisy across all layers, the signal may be too weak to be useful, which would be an
important (if less exciting) finding in itself. Send me this table.

## 3. Smoke test — trace one image through InfoPrune

In [34]:
import gc
import torch

if 'model' in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

In [35]:
enable_infoprune()

fastv_config_adaptive = {
    "use_fastv": True,
    "fastv_r": 0.5,
    "image_token_start_index": image_start,
    "image_token_length": image_len,
    "layer_floor": LAYER_FLOOR,
    "layer_ceiling": LAYER_CEILING,
    "dataset_entropy": dataset_entropy,
    "layer_calib_scores": layer_calib_scores,
}

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, attn_implementation="eager", device_map="auto",
    fastv_config=fastv_config_adaptive,
)
prompt = "USER: <image>\nWhat is shown in this image? Answer in one sentence.\nASSISTANT:"
image = Image.open(requests.get("http://images.cocodataset.org/val2017/000000039769.jpg", stream=True).raw)
inputs = processor(prompt, image, return_tensors="pt").to(model.device)

_LAST_TRACE["chosen_layer"] = None
_LAST_TRACE["trace"] = None
out = model.generate(**inputs, max_new_tokens=64, do_sample=False, use_cache=False)
print("INFOPRUNE OUTPUT:", processor.decode(out[0], skip_special_tokens=True))
print("Chosen layer:", _LAST_TRACE["chosen_layer"])
print("Trace (layer, H_img, InfoScore):", _LAST_TRACE["trace"])

del model
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

INFOPRUNE OUTPUT: USER:  <image> 
What is shown in this image? Answer in one sentence.
ASSISTANT: Two cats sleeping on a couch.
Chosen layer: 5
Trace (layer, H_img, InfoScore): {2: 0.5441, 3: 0.3529, 4: 0.5, 5: 0.6765, 6: 0.1765, 7: 0.1029, 8: 0.5, 9: 0.1618, 10: 0.1912, 11: 0.5882, 12: 0.2059, 13: 0.3676, 14: 0.1765, 15: 0.1471, 16: 0.3088}


In [36]:
# Run a handful more images with different content to see whether the chosen layer actually varies --
# this is the whole premise of the method, so worth checking on more than one example before the full sweep.
demo_urls = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",  # Sleeping cats
    "http://images.cocodataset.org/val2017/000000000139.jpg",  # Living room / fireplace
    "http://images.cocodataset.org/val2017/000000000285.jpg",  # Bear in woods
    "http://images.cocodataset.org/val2017/000000000724.jpg",  # upside down Stop sign 
    "http://images.cocodataset.org/val2017/000000001000.jpg",  # Tennis players
]
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, attn_implementation="eager", device_map="auto",
    fastv_config=fastv_config_adaptive,
)
for url in demo_urls:
    image = Image.open(requests.get(url, stream=True).raw)
    inputs = processor(prompt, image, return_tensors="pt").to(model.device)
    _LAST_TRACE["chosen_layer"] = None
    _LAST_TRACE["trace"] = None
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=1, do_sample=False, use_cache=False)
    print(url.split('/')[-1], "-> chosen layer:", _LAST_TRACE["chosen_layer"])
del model
torch.cuda.empty_cache()
disable_infoprune()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

000000039769.jpg -> chosen layer: 5
000000000139.jpg -> chosen layer: 4
000000000285.jpg -> chosen layer: 14
000000000724.jpg -> chosen layer: 3
000000001000.jpg -> chosen layer: 2


## 4. Full comparison on the POPE subset

Runs baseline, fixed-K FastV (K=2, R=0.5 — your best Phase-0 setting), and InfoPrune (R=0.5,
adaptive layer) on the same POPE questions from Phase 0, plus records the chosen-layer distribution
for InfoPrune so we can see whether adaptivity is real.

In [37]:
import gc
from collections import Counter

# Flatten the subset dictionary into a single list of questions
all_questions = [q for split_data in subset.values() for q in split_data]
def parse_yes_no(text):
    text = text.split("ASSISTANT:")[-1].strip().lower()
    if text.startswith("yes"):
        return "yes"
    if text.startswith("no"):
        return "no"
    # fallback: search for first yes/no token
    m = re.search(r"\b(yes|no)\b", text)
    return m.group(1) if m else "no"
def pope_metrics(preds, labels):
    tp = sum(p == "yes" and l == "yes" for p, l in zip(preds, labels))
    fp = sum(p == "yes" and l == "no" for p, l in zip(preds, labels))
    tn = sum(p == "no" and l == "no" for p, l in zip(preds, labels))
    fn = sum(p == "no" and l == "yes" for p, l in zip(preds, labels))
    acc = (tp + tn) / len(labels)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    yes_ratio = preds.count("yes") / len(preds)
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, yes_ratio=yes_ratio)


def run_eval_infoprune(fastv_config, tag, track_layers=False):
    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, attn_implementation="eager", device_map="auto",
        fastv_config=fastv_config,
    )
    model.eval()
    preds, labels, chosen_layers = [], [], []
    for i, q in enumerate(all_questions):
        img = Image.open(img_dir / q["image"]).convert("RGB")
        prompt = f"USER: <image>\n{q['text']}\nASSISTANT:"
        inputs = processor(prompt, img, return_tensors="pt").to(model.device)
        if track_layers:
            _LAST_TRACE["chosen_layer"] = None
            _LAST_TRACE["trace"] = None
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=10, do_sample=False, use_cache=False)
        pred = parse_yes_no(processor.decode(out[0], skip_special_tokens=True))
        preds.append(pred)
        labels.append(q["label"])
        if track_layers and _LAST_TRACE["chosen_layer"] is not None:
            chosen_layers.append(_LAST_TRACE["chosen_layer"])
        if (i + 1) % 50 == 0:
            print(f"  [{tag}] {i+1}/{len(all_questions)}")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return preds, labels, chosen_layers

results = {}
layer_dists = {}

In [ ]:
print("=== Baseline ===")
disable_infoprune()
preds, labels, _ = run_eval_infoprune(None, "baseline")
results["baseline"] = pope_metrics(preds, labels)

=== Baseline ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
import importlib
from transformers.models.llama import modeling_llama

# Hard-reset the module to its original unpatched state
importlib.reload(modeling_llama) 

print("=== FastV K=2 R=0.5 (fixed) ===")
fixed_cfg = {
    "use_fastv": True, 
    "fastv_k": 2, 
    "fastv_r": 0.5,
    "image_token_start_index": image_start, 
    "image_token_length": image_len
}

# Run the evaluation
preds, labels, _ = run_eval_infoprune(fixed_cfg, "fastv_fixed")
results["FastV K=2 R=0.5"] = pope_metrics(preds, labels)

# Important: If you need InfoPrune for the next cell, reapply the patch!
# enable_infoprune()

In [ ]:
print("=== InfoPrune R=0.5 (adaptive) ===")
enable_infoprune()
preds, labels, chosen = run_eval_infoprune(fastv_config_adaptive, "infoprune", track_layers=True)
results["InfoPrune R=0.5"] = pope_metrics(preds, labels)
layer_dists["InfoPrune R=0.5"] = Counter(chosen)
disable_infoprune()

In [ ]:
print("\n\n==== POPE COMPARISON ====")
print(f"{'setting':<22}{'acc':>8}{'f1':>8}{'yes%':>8}")
for tag, m in results.items():
    print(f"{tag:<22}{m['accuracy']*100:>7.1f}%{m['f1']*100:>7.1f}%{m['yes_ratio']*100:>7.1f}%")

In [ ]:
print("\n==== INFOPRUNE CHOSEN-LAYER DISTRIBUTION ====")
for tag, dist in layer_dists.items():
    print(tag, "->", dict(sorted(dist.items())))

**What to send back:**
1. The `H_data(l)` table from Section 2
2. Whichever chosen-layer traces you get from the single-image check in Section 3
3. The `==== POPE COMPARISON ====` table and the chosen-layer distribution from Section 4

**What we're looking for:** does InfoPrune match or beat FastV's fixed K=2/R=0.5 accuracy, and does
the chosen-layer distribution actually spread across multiple layers (not collapse to one value)? If
it collapses to a single layer, that's informative too — it would mean, for this task/model, a fixed
K really is close to optimal, and the paper's honest framing shifts toward "adaptive selection matches
a well-tuned fixed baseline without needing the tuning" rather than "adaptive selection beats it" —
still a defensible contribution, just a different claim. Once you have these numbers, we can decide
whether to tune `TARGET_AVG_LAYER`/`info_threshold` further or move to Phase 3 (integrating SparseVLM's
text-token filtering) and a wider benchmark sweep.